# 04. Machine Learning Ranking Model (Learning-to-Rank with XGBoost)

This notebook demonstrates the Learning-to-Rank (LTR) stage:
1. **Feature Engineering**: Extracting 34 query-listing, price, quality, host, and spatial features.
2. **Synthetic Relevance Labelling**: Formulating multi-grade relevance criteria (disclosed as simulated data).
3. **Query-Level Split**: Partitioning queries 80/20 to prevent data leakage.
4. **XGBRanker Training**: Optimizing `rank:ndcg` with LambdaMART/Hist gradient boosting.
5. **Feature Importance & SHAP Analysis**: Inspecting feature contributions and explainability.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import config
from src.search import AirbnbSearchEngine
from src.features import FeatureEngineer, FEATURE_NAMES
from src.ranking import (
    generate_queries,
    build_training_dataset,
    query_level_split,
    train_ranker,
    XGBRankingModel,
    save_ranker,
)
from evaluation.evaluate import SearchEvaluator, print_results_table

## 1. Load Search Engine & Initialize Feature Engineer

In [ ]:
engine = AirbnbSearchEngine()
engine.load()
fe = FeatureEngineer(engine.listings)
print(f"Feature Engineer initialized with {len(FEATURE_NAMES)} engineered ranking features.")

## 2. Generate Multi-City Queries & Candidate Dataset
*(If datasets are already saved in `models/ranker/`, we load them directly)*

In [ ]:
if config.TRAIN_FEATURES_PATH.exists() and config.TEST_FEATURES_PATH.exists():
    train_df = pd.read_parquet(config.TRAIN_FEATURES_PATH)
    test_df = pd.read_parquet(config.TEST_FEATURES_PATH)
    print(f"Loaded cached feature sets: Train ({len(train_df):,} pairs), Test ({len(test_df):,} pairs)")
else:
    queries = generate_queries(n=config.NUM_QUERIES)
    dataset = build_training_dataset(queries, engine, fe)
    train_df, test_df = query_level_split(dataset)
    train_df.to_parquet(config.TRAIN_FEATURES_PATH, index=False)
    test_df.to_parquet(config.TEST_FEATURES_PATH, index=False)

print(f"Train Queries: {train_df['query_id'].nunique()}, Test Queries: {test_df['query_id'].nunique()}")
print(f"Relevance Label Distribution:\n{train_df['relevance'].value_counts(normalize=True).round(3)}")

## 3. Train XGBRanker Model

In [ ]:
ranker = train_ranker(train_df, test_df)
save_ranker(ranker, FEATURE_NAMES)

## 4. Feature Importance Analysis

In [ ]:
importance = ranker.get_booster().get_score(importance_type="gain")
imp_df = pd.DataFrame(list(importance.items()), columns=["feature", "gain"]).sort_values("gain", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=imp_df.head(15), x="gain", y="feature", palette="crest")
plt.title("Top 15 Feature Importances by Gain (XGBRanker)", fontsize=14, fontweight="bold")
plt.xlabel("Average Gain")
plt.ylabel("Feature Name")
plt.show()

## 5. End-to-End Evaluation Across All 4 Systems
Compares: TF-IDF vs. Semantic Search vs. Hybrid vs. Hybrid + XGBoost

In [ ]:
evaluator = SearchEvaluator()
evaluator.load()
results = evaluator.run_all()
print_results_table(results)